# Table 4 — Adjuster Capacity
## Purpose
A snapshot of each adjuster's current workload and availability. Used to determine
whether a new claim can be routed to a given adjuster or whether they are at/over capacity.

## Columns
| Column | Description |
|--------|-------------|
| Adjuster ID / Name | Matches Table 3 |
| Active Open Claims | Current number of open claims (from Table 3) |
| Max Claim Capacity | Role-based upper limit set manually (see below) |
| Weighted Workload Score | Capacity-adjusted load accounting for claim complexity |
| Availability Status | Derived from % utilisation and PTO flag |
| PTO/OOO Flag | `Y` = adjuster is currently out of office |

## Weighted Workload Score — how it is calculated
Each open claim contributes a different number of points based on complexity:

| Complexity | Points |
|-----------|--------|
| High      | 3      |
| Medium    | 2      |
| Low       | 1      |

Since individual claim complexity data is not available per adjuster, a simplified
mix is assumed across their portfolio: **20% High, 40% Medium, 40% Low**.

Formula:
```
score = (active * 0.20 * 3) + (active * 0.40 * 2) + (active * 0.40 * 1)
      = active * (0.6 + 0.8 + 0.4)
      = active * 1.8  (approximately)
```

## Availability Status thresholds
| Utilisation (active / max) | Status |
|---------------------------|--------|
| < 60% | Available |
| 60–79% | Near Capacity |
| 80–99% | At Capacity |
| >= 100% | Overloaded |
| PTO = Y | Unavailable - PTO (overrides utilisation) |

## Max Claim Capacity — rationale by role
Capacity limits reflect role expectations in the insurance industry:
- **Adjuster I** — 50–55 claims (lower complexity, higher volume)
- **Adjuster II / Specialist** — 40–60 claims (moderate complexity)
- **CAT Specialist** — 90 claims (surges during events, short-lived)
- **WC Specialist** — 65–70 claims (long-tail claims, steady load)
- **SIU Lead** — 20 claims (each case is complex and investigative)
- **Senior Supervisor** — 15 claims (oversight role, not primary handler)
- **Subrogation Specialist** — 30 claims (recovery-focused, moderate volume)

In [1]:
import pandas as pd
import random

# Load Table 3 — capacity is derived from adjuster profiles
df_profiles = pd.read_csv("table3_adjuster_profile_capability.csv").fillna("")
print(f"Loaded Table 3: {len(df_profiles)} adjusters")

Loaded Table 3: 15 adjusters


## Max claim capacity by adjuster

In [2]:
# Hand-set per adjuster based on role (see rationale in header above)
max_capacities = {
    "ADJ-001": 50,   # Claims Adjuster II — Auto
    "ADJ-002": 55,   # Claims Adjuster I  — Auto (higher volume, lower complexity)
    "ADJ-003": 40,   # Liability Specialist — complex, litigation-heavy
    "ADJ-004": 45,   # Property Adjuster II
    "ADJ-005": 65,   # WC Specialist — long-tail, high volume
    "ADJ-006": 20,   # SIU Lead — investigative, low volume
    "ADJ-007": 55,   # Claims Adjuster I — Auto
    "ADJ-008": 50,   # Property Adjuster I
    "ADJ-009": 90,   # CAT Specialist — surge capacity during events
    "ADJ-010": 35,   # Liability Specialist — high-value BI, low volume
    "ADJ-011": 55,   # Material Damage Specialist
    "ADJ-012": 70,   # WC Adjuster II — vocational rehab, high volume
    "ADJ-013": 50,   # Material Damage Specialist
    "ADJ-014": 15,   # Senior Supervisor — oversight only
    "ADJ-015": 30,   # Subrogation Specialist
}

## Weighted Workload Score function

In [3]:
def calc_weighted_workload(n):
    """
    Approximate weighted workload score for an adjuster with n active claims.

    Assumes portfolio mix: 20% High (3 pts), 40% Medium (2 pts), 40% Low (1 pt).

    Example for n=50:
        High  = round(50 * 0.20) = 10  →  10 * 3 = 30
        Medium= round(50 * 0.40) = 20  →  20 * 2 = 40
        Low   = 50 - 10 - 20    = 20  →  20 * 1 = 20
        Score = 30 + 40 + 20 = 90
    """
    h = round(n * 0.20)
    m = round(n * 0.40)
    l = n - h - m
    return h * 3 + m * 2 + l * 1

## Availability Status function

In [4]:
def get_availability(active, max_cap, pto):
    """
    PTO overrides utilisation.
    Thresholds: <60% Available, 60-79% Near Capacity,
                80-99% At Capacity, >=100% Overloaded.
    """
    if pto == "Y":
        return "Unavailable - PTO"
    pct = active / max_cap
    if pct < 0.60: return "Available"
    if pct < 0.80: return "Near Capacity"
    if pct < 1.00: return "At Capacity"
    return "Overloaded"

## Generate capacity rows

In [5]:
random.seed(42)

capacity_rows = []
for _, p in df_profiles.iterrows():
    adj_id  = p["Adjuster ID"]
    active  = int(p["Active Claim Count"])
    max_cap = max_capacities[adj_id]
    # 10% chance of being on PTO at any given snapshot
    pto = random.choices(["Y", "N"], weights=[0.10, 0.90])[0]

    capacity_rows.append({
        "Adjuster ID":            adj_id,
        "Adjuster Name":          p["Adjuster Name"],
        "Active Open Claims":     active,
        "Max Claim Capacity":     max_cap,
        "Weighted Workload Score": calc_weighted_workload(active),
        "Availability Status":    get_availability(active, max_cap, pto),
        "PTO/OOO Flag":           pto,
    })

df_capacity = pd.DataFrame(capacity_rows)
df_capacity.to_csv("table4_adjuster_capacity.csv", index=False)
print(f"Table 4: {len(df_capacity)} rows saved to table4_adjuster_capacity.csv")
df_capacity

Table 4: 15 rows saved to table4_adjuster_capacity.csv


,Adjuster ID,Adjuster Name,Active Open Claims,Max Claim Capacity,Weighted Workload Score,Availability Status,PTO/OOO Flag
0,ADJ-001,Sarah Mitchell,38,50,69,Near Capacity,N
1,ADJ-002,James Kowalski,42,55,75,Unavailable - PTO,Y
2,ADJ-003,Linda Reyes,28,40,51,Near Capacity,N
3,ADJ-004,Marcus Thompson,31,45,55,Near Capacity,N
4,ADJ-005,Patricia Chen,55,65,99,At Capacity,N
5,ADJ-006,David O'Brien,15,20,27,Near Capacity,N
6,ADJ-007,Anita Patel,47,55,84,At Capacity,N
7,ADJ-008,Robert Harrington,36,50,64,Unavailable - PTO,Y
8,ADJ-009,Kevin Walsh,82,90,147,At Capacity,N
9,ADJ-010,Jennifer Nguyen,22,35,39,Unavailable - PTO,Y


## Validation — utilisation distribution

In [6]:
df_capacity["Utilisation %"] = (
    df_capacity["Active Open Claims"] / df_capacity["Max Claim Capacity"] * 100
).round(1)

print("Availability breakdown:")
print(df_capacity["Availability Status"].value_counts().to_string())
print()
print("\nWorkload snapshot:")
cols = ["Adjuster Name", "Active Open Claims", "Max Claim Capacity",
        "Utilisation %", "Weighted Workload Score", "Availability Status"]
print(df_capacity[cols].to_string(index=False))

Availability breakdown:
Availability Status
Near Capacity        6
At Capacity          5
Unavailable - PTO    4


Workload snapshot:
    Adjuster Name  Active Open Claims  Max Claim Capacity  Utilisation %  Weighted Workload Score Availability Status
   Sarah Mitchell                  38                  50           76.0                       69       Near Capacity
   James Kowalski                  42                  55           76.4                       75   Unavailable - PTO
      Linda Reyes                  28                  40           70.0                       51       Near Capacity
  Marcus Thompson                  31                  45           68.9                       55       Near Capacity
    Patricia Chen                  55                  65           84.6                       99         At Capacity
    David O'Brien                  15                  20           75.0                       27       Near Capacity
      Anita Patel                  47   